In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/02.bronze-helpers

In [0]:
# Define source_file and table_name
source_file = f"{landing_folder_path}/{v_batch_id}/drivers.json"
table_name = f"{catalog_name}.{bronze_schema}.drivers"

**Step 1 - Read the CSV file using the dataframe reader API**

In [0]:
# Define the schema
from pyspark.sql.types import StructType, StructField, StringType, DateType

name_schema = StructType([
    StructField("givenName", StringType()),
    StructField("familyName", StringType())
])

drivers_schema = StructType([
    StructField("driverId", StringType()),
    StructField("name", name_schema),
    StructField("dateOfBirth", DateType()),
    StructField("nationality", StringType()),
    StructField("url", StringType())
])

In [0]:
drivers_df = (
    spark.read
         .format("json")
         .schema(drivers_schema)
         .load(source_file)
)

**Step 2 - Add Metadata Columns**

- Source File
- Ingestion Timestamp

In [0]:
drivers_final_df = add_ingestion_metadata(drivers_df)

**Step 3 - Write to bronze delta table**

In [0]:
write_to_bronze(
    input_df = drivers_final_df,
    target_table = table_name,
    batch_id = v_batch_id
)

In [0]:
%sql
select * from formula1_incr.bronze.drivers